In [1]:
import os
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Optional

In [2]:
#PATHS
RAW_DATA_ROOT = "data/raw/"
MASTER_PATH =  "file_info/files_master.csv"
#files to inspect
n_samples = 15

In [4]:
files_master = pd.read_csv(MASTER_PATH)
#we only care about v1 for now
v1 = files_master[files_master["include_v1"] == True].copy()

print(f"Total files: {len(files_master)}")
print(f"files in v1: {len(v1)}")
print(f"labels (v1):")
print(v1["label_binary"].value_counts().rename({0: "control", 1: "disorder"}))


#sample files
sample = (v1.groupby("label_binary", group_keys = False).sample(n=2, random_state=42).head(n_sample))

FileNotFoundError: [Errno 2] No such file or directory: 'file_info/files_master.csv'

In [ ]:
#parse cha file
def read_raw_file(file_path: str, root=RAW_DATA_ROOT, n_lines = 30) -> tuple[Optional[list[str]], Optional[str]]:

  full_path = os.path.join(root, file_path)

  try:
    #utf8 encoding first
    with open(full_path, encoding="utf-8") as file:
      lines = file.readlines()
      return lines, None
  #probably not needed to add anything else, can add later as needed
  except Exception as e:
      return None, str(e)

In [ ]:
for idx, row in sample.iterrows():

  lines, error = read_raw_file(row("file_path"))

  #if is broken/didn't work
  if error:
    print(f"Error reading file {row['file_path']}: {error}")
  if lines is None:
    print(f"No lines found for file {row['file_path']}")
    continue

#if did work afterwards, get that info
for idx, line in enumerate(lines[:30]):
  print(f"{idx+1:3d} {line}", end ="")


In [ ]:
#process headers

def process_headers(lines: list[str]) -> dict:

  headers = {}

  for line in lines:
    line = line.strip()
    #all info lines seem to start with @ but that could be wrong, just manual brief inspection determined this
    if line.startswith("@") and ":" in line:
      key, ignore, val = line.partition(":")
      key = key.strip()
      val = val.strip()
      headers[key] = val
    elif line.startswith("*"):
      #this is utterance
      break

  return headers



In [ ]:
##example line  type:
# *INV:	tell me about the school holiday .
# %mor:	v|tell pro:obj|me prep|about det:art|the n|school n|holiday .
# %gra:	1|0|ROOT 2|1|OBJ 3|1|JCT 4|6|DET 5|6|MOD 6|3|POBJ 7|1|PUNCT

In [ ]:
def raw_utterances(lines: list[str]) -> list[str]:
  """
  Extracts the raw utterances, all starting with *. No cleaning at all
  """
  utterances = []
  current = None
  end_field = None

  for line in lines:
    line = line.rstrip("\n")
    if line.startswith("*"):
      if current:
        utterances.append(current)

      #regex to match cases where *[insert speaker]: [insert words said]
      match_speech = re.match_speech(r'^\*([A-Z0-9]+):\t?(.*)', line)
      if match_speech:
        current = {"speaker": match_speech.group(1), "text": match_speech.group(2), "oth_part": {}}
        end_field = "text"
      else:
        current = {"speaker": None, "text": line, "oth_part": {}, "parse_warning": True}
        last_field = "text"

    elif line.startswith("%") and current is not None:
      match_line = re.match_line(r'^%*[^:]+):\t?(.*)',line)
      if match_line:
        oth_name = match_line.group(1)
        current["oth_part"][oth_name] = match_line.group(2)
        last_field = oth_name

    elif line.startswith("\t") and current is not None:
      continuation = line.strip()
      if end_field == "text":
        current[end_field] += " " + continuation
      elif last_field in current["oth_part"]:
        current["oth_part"][last_field] += " " + continuation

  if current:
    utterances.append(current)
  return utterances



In [ ]:
speakers_counts = defaultdict(int)
oth_lines_counts = defaultdict(int)
errors = []
files = []

for idx, row in sample.iterrows():
    lines, error = read_raw_file(row["file_path"])
    if lines is None:
        continue

    utterances = raw_utterances(lines)

    speakers = set()
    error_count = 0
    file_oth_names = set()

    for speech in utterances:
        speakers_counts[speech["speaker"]] += 1
        speakers.add(speech["speaker"])
        files.append(row["file_id"])

        for oth_name in speech["oth_part"]:
            oth_lines_counts[oth_name] += 1
            file_oth_names.add(oth_name)

        if speech.get("parse_warning"):
            error_count += 1
            errors.append({
                "file_id": row["file_id"],
                "line": speech["text"]
            })

    print(f"\nfile_id: {row['file_id']}")
    print(f"total utterances: {len(utterances)}")
    print(f"speakers found: {sorted(speakers)}")
    print(f"other lines present:{sorted(file_oth_names)}")
    if error_count:
        print(f"warning:     {error_count} line didnt look as expected")

print("\nspeaker totals in sample:")
for code, count in sorted(speakers_counts.items(), key=lambda x: -x[1]):
    print(f"  *{code}: {count} utterances")

print("\nother lines totals in sample:")
for oth_name, count in sorted(oth_lines_counts.items(), key=lambda x: -x[1]):
    print(f"%{oth_name}: {count} occurrences")

In [ ]:
chat_codes = {
    "pause_short":        r"\(\.\)",
    "pause_medium":       r"\(\.\.\)",
    "pause_long":         r"\(\.\.\.\)",
    "pause_timed":        r"\(\d+\.?\d*\)",
    "filled_pause":       r"&(uh|um|er)\b",
    "repetition":         r"\[/\]",
    "revision":           r"\[//\]",
    "reformulation":      r"\[///\]",
    "error_marker":       r"\[\*\]",
    "target_form":        r"\[:",
    "unintelligible_xxx": r"\bxxx\b",
    "unintelligible_yyy": r"\byyy\b",
    "paralinguistic":     r"&=\w+",
    "uncertain":          r"\[\?\]",
    "trailing_off":       r"\+\.\.\.",
    "interrupted":        r"\+/\.",
}

code_totals = defaultdict(lambda: {"count": 0, "files": 0})

for idx, row in sample.iterrows():
    lines, error = read_raw_file(row["file_path"])
    if lines is None:
        continue

    utterances = raw_utterances(lines)
    utterance_text = " ".join(speech["text"] for speech in utterances if speech.get("text"))

    for code, pattern in chat_codes.items():
        matches = re.findall(pattern, utterance_text)
        if matches:
            code_totals[code]["count"] += len(matches)
            code_totals[code]["files"] += 1

print("chat codes summary")
print(f"{'code':<22} {'total':>12} {'files':>16}")


for code_name, stats in sorted(code_totals.items(), key=lambda x: -x[1]["count"]):
    print(
        f"{code_name:<22} "
        f"{stats['count']:>12} "
        f"{stats['files']:>11}/{len(sample)}"
    )

zero_codes = [code for code in chat_codes if code_totals[code]["count"] == 0]

print("\ncodes not seen:")
if zero_codes:
    for code in zero_codes:
        print(f"- {code}")
else:
    print("None")